# 14 — Source review and alert workload

**Purpose:** finish the two practical checks before user evaluation, without another model experiment.

After your completed longer-history export in 08, **Restart Kernel and Run All here**. This notebook reads the frozen longer-history **validation** predictions. It never loads test rows, fits a model, chooses a winning policy or changes the dashboard/November registration.

1. Prepare 12 high-priority training-price cases for independent investigation.
2. Compare the unchanged alert rule with illustrative weekly review capacities of 25, 50 and 100 cases.

This is post-hoc exploratory analysis on an already inspected validation period. Better precision is not guaranteed, and reduced workload can miss more surges. Review budgets need stakeholder input before adoption.


In [1]:
# Works from the FYP folder or research/.
import sys
from pathlib import Path
repository = Path.cwd()
if not (repository / "pricecatcher_config.py").is_file():
    repository = repository.parent
if not (repository / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open the FYP project folder before running this notebook.")
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))

import hashlib
import json
from importlib import reload
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
import pricecatcher_config as cfg
cfg = reload(cfg)


def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def verify(path, expected):
    if not path.is_file() or file_hash(path) != expected:
        raise ValueError(f"Frozen input missing/changed: {path}. Restore its original file; nothing is regenerated here.")


print("Run folder:", cfg.PROJECT_ROOT)


Run folder: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240


## 1. Verify the chosen checkpoint and validation population

Use the exact longer-history checkpoint already published by 08, rather than selecting another experiment by its scores. Hashes identify the checkpoint and validation files. Saved prediction keys/labels must match the frozen validation population and its original chronological boundaries.


In [2]:
entry_path = cfg.PROJECT_ROOT / "08_dashboard" / "dashboard_manifest.json"
if not entry_path.is_file():
    raise FileNotFoundError("Finish the longer-history export in Notebook 08 first. Do not restart 01.")
entry = json.loads(entry_path.read_text())
model = entry["forecast_model"]
if model["version"] != "longer_2024_2025":
    raise ValueError("This audit is for your completed longer-history candidate. Complete research 12 and export it in 08 first.")
checkpoint = (cfg.PROJECT_ROOT / model["model_path"]).resolve()
if not checkpoint.is_relative_to(cfg.PROJECT_ROOT.resolve()):
    raise ValueError("Checkpoint is outside the selected run.")
verify(checkpoint, model["sha256"])
experiment_path = checkpoint.parent / "experiment_manifest.json"
experiment = json.loads(experiment_path.read_text())
if experiment.get("prepared_test_accessed") is not False or experiment["protocol"]["cutoff"] != model["score_cutoff"]:
    raise ValueError("Longer-history experiment/cutoff disagrees with the dashboard.")
verify(checkpoint, experiment["output_sha256"][checkpoint.name])
folder = cfg.PROJECT_ROOT / "experiments" / "holdout" / "05_modelling"
for name in ["experiment_settings.json", "validation.parquet"]:
    verify(folder / name, experiment["protocol"]["input_sha256"][str((folder / name).resolve())])
settings = json.loads((folder / "experiment_settings.json").read_text())
cutoff = float(model["score_cutoff"])
keys = cfg.GROUP_KEYS + ["week_start"]
prediction_path = checkpoint.parent / "validation_predictions.parquet"
verify(prediction_path, experiment["output_sha256"][prediction_path.name])
validation = pd.read_parquet(prediction_path, columns=keys + ["next_week_end", "surge_label", "longer_2024_2025_score"])
validation = validation.rename(columns={"longer_2024_2025_score": "model_score"})
frozen = pd.read_parquet(folder / "validation.parquet", columns=keys + ["next_week_end", "surge_label"])
if validation.empty or validation[keys].isna().any().any() or validation.duplicated(keys).any() or frozen.duplicated(keys).any():
    raise ValueError("Validation identifiers must be nonempty, unique and complete.")
matched = frozen.merge(validation, on=keys, how="outer", suffixes=("_frozen", ""), validate="one_to_one", indicator=True)
if (not matched["_merge"].eq("both").all() or not matched.surge_label_frozen.eq(matched.surge_label).all()
        or not matched.next_week_end_frozen.eq(matched.next_week_end).all()):
    raise ValueError("Saved validation predictions do not reproduce the frozen population/labels.")
if (not validation.model_score.between(0, 1).all() or not validation.surge_label.isin([0, 1]).all() or validation.surge_label.nunique() != 2
        or not validation.week_start.dt.dayofweek.eq(0).all()
        or not validation.next_week_end.eq(validation.week_start + pd.Timedelta(days=13)).all()
        or not (validation.week_start + pd.Timedelta(days=6)).gt(pd.Timestamp(settings["train_end"])).all()
        or not validation.next_week_end.le(pd.Timestamp(settings["validation_end"])).all()):
    raise ValueError("Invalid scores, labels or validation dates.")
print("Verified validation rows:", len(validation), "| Fixed cutoff:", cutoff)
print("No test rows, model objects or future-period data were loaded.")


Verified validation rows: 43594 | Fixed cutoff: 0.6006503045679725
No test rows, model objects or future-period data were loaded.


## 2. Prepare the source investigation log

This checks that shortlisted prices/units match the verified integrated snapshot. It **does not establish whether the prices are economically correct**. Nearby observations and a fresh download of the same record are not independent confirmation.

Seek a provider correction, an independently dated shop record/receipt, or another authoritative record for the same item, unit, shop and date. Record the reference, notes, reviewer and date in the generated `source_review_log.csv`. Decisions are user-recorded evidence, not independently authenticated by this notebook. The log is preserved across reruns and never drives automatic corrections. Leave unresolved cases unresolved.


In [3]:
source = cfg.verify_source_snapshot()
snapshot = source["source_snapshot_sha256"]
if settings["preparation"]["source_snapshot_sha256"] != snapshot:
    raise ValueError("Source review belongs to another prepared snapshot.")
review_path = cfg.PROJECT_ROOT / "experiments" / "holdout" / "03_cleaned" / "training_source_context.csv"
if not review_path.is_file():
    raise FileNotFoundError("Run only the final source-review cell in Notebook 03, then rerun 14. Do not rerun preparation.")
review = pd.read_csv(review_path, parse_dates=["date"])
if (review.empty or review.case_id.duplicated().any() or review.date.isna().any()
        or review.date.gt(pd.Timestamp(settings["train_end"])).any()
        or not review.source_snapshot_sha256.eq(snapshot).all()):
    raise ValueError("Review cases must have unique IDs and belong to this training snapshot only.")
# A review priority, not an exclusion rule or suggested replacement price.
ratios = review[["ratio_to_previous", "ratio_to_following"]]
review["deviation"] = np.log(ratios.where(ratios.gt(0))).abs().max(axis=1)
shortlist = review.sort_values(["review_priority", "deviation", "case_id"], ascending=[True, False, True]).head(12).copy()
raw = pd.read_parquet(cfg.STAGE1_COMBINED_PARQUET, columns=["date", "premise_code", "item_code", "price", "unit"],
    filters=[("date", "in", shortlist.date.tolist()), ("premise_code", "in", shortlist.premise_code.tolist()),
             ("item_code", "in", shortlist.item_code.tolist())])
checked = shortlist.merge(raw, on=["date", "premise_code", "item_code"], how="left",
                          suffixes=("", "_snapshot"), validate="one_to_one", indicator=True)
if (not checked["_merge"].eq("both").all() or not checked.unit.eq(checked.unit_snapshot).all()
        or not np.isclose(checked.price, checked.price_snapshot, rtol=0, atol=1e-12, equal_nan=True).all()):
    raise ValueError("Shortlisted prices/units differ from the verified source snapshot.")
review_dir = cfg.PROJECT_ROOT / "14_review_readiness" / "source_review" / snapshot[:16]
review_dir.mkdir(parents=True, exist_ok=True)
log_path = review_dir / "source_review_log.csv"
fields = ["case_id", "date", "project_group", "item", "price", "unit", "premise_code", "item_code", "premise",
          "state", "district", "previous_price", "following_price", "review_priority", "source_month_url", "source_snapshot_sha256"]
new_cases = shortlist[fields].copy()
new_cases["decision"] = "unresolved"
for name in ["evidence_reference", "evidence_notes", "reviewer", "reviewed_on"]:
    new_cases[name] = ""
if log_path.exists():
    log = pd.read_csv(log_path, keep_default_na=False)
    if log.case_id.duplicated().any() or not log.case_id.isin(review.case_id).all() or not log.source_snapshot_sha256.eq(snapshot).all():
        raise ValueError("Review log has duplicate IDs or belongs to another snapshot; original notes were not overwritten.")
    additions = new_cases.loc[~new_cases.case_id.isin(log.case_id)]
    if not additions.empty:
        log = pd.concat([log, additions], ignore_index=True)
        log.to_csv(log_path, index=False)
else:
    log = new_cases
    log.to_csv(log_path, index=False)
if not log.decision.isin(["unresolved", "confirmed_error", "confirmed_valid"]).all():
    raise ValueError("Use decision unresolved, confirmed_error or confirmed_valid in the log.")
claimed = log.decision.ne("unresolved")
for name in ["evidence_reference", "evidence_notes", "reviewer", "reviewed_on"]:
    if log.loc[claimed, name].astype(str).str.strip().eq("").any():
        raise ValueError("A recorded confirmation needs its reference, notes, reviewer and review date. No prices were changed.")
if pd.to_datetime(log.loc[claimed, "reviewed_on"], errors="coerce").isna().any():
    raise ValueError("A recorded confirmation needs a valid review date, preferably YYYY-MM-DD.")
display(shortlist[["case_id", "project_group", "price", "unit", "previous_price", "following_price", "review_priority"]])
print("Review log (preserved on reruns):", log_path)
print("Log decisions are user-recorded, not independently verified by this notebook:")
display(log.groupby("decision").size().rename("cases").to_frame())
print("Matching the source snapshot confirms integration consistency, not the correctness of a source price.")


,case_id,project_group,price,unit,previous_price,following_price,review_priority
52,2025-06-17_17152_1556,Vegetables,6399.00,1kg,6.99,6.990000,1
43,2025-07-23_14863_114,Vegetables,0.03,1kg,5.29,5.290000,1
29,2025-03-02_2_93,Vegetables,1600.00,1kg,16.00,15.000000,1
7,2025-10-13_21223_1109,Eggs,500.00,10 biji,5.00,4.766667,1
12,2025-02-17_5093_1111,Eggs,370.00,10 biji,3.70,3.550000,1
56,2025-01-06_2130_1558,Vegetables,380.00,1kg,3.80,4.000000,1
54,2025-01-16_20759_1556,Vegetables,399.00,1kg,3.99,3.990000,1
18,2025-01-20_11621_917,Flour,318.00,1kg,3.20,3.180000,1
5,2025-01-26_2007_118,Eggs,420.00,10 biji,4.20,4.200000,1
57,2025-03-15_14824_1558,Vegetables,349.00,1kg,3.49,3.490000,1


Review log (preserved on reruns): /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/14_review_readiness/source_review/1866ba0fc10bebdc/source_review_log.csv
Log decisions are user-recorded, not independently verified by this notebook:


,cases
decision,
unresolved,12


Matching the source snapshot confirms integration consistency, not the correctness of a source price.


## 3. Compare a few review workloads, without selecting a winner

The reference uses the frozen score cutoff. Each alternative reviews **at most 25, 50 or 100 of those alerts per calendar week**, across the full original validation population. Scores below cutoff are never promoted. Equal-score ties use the existing series keys deterministically; tie order is not predictive evidence.

These are illustrative national-study review capacities, not validated user requirements or new classification cutoffs. The dashboard's Spotlight/filter list is not automatically assigned this performance. Report false alerts and missed surges together. AP is unchanged because the model scores are unchanged. Group results reveal whether a short list concentrates on vegetables.


In [4]:
def policy_metrics(frame, predicted):
    y = frame.surge_label.eq(1)
    tp = int((y & predicted).sum())
    fp = int((~y & predicted).sum())
    fn = int((y & ~predicted).sum())
    positives = int(y.sum())
    return {"rows": len(frame), "positives": positives, "review_cases": tp + fp,
            "true_alerts": tp, "false_alerts": fp, "missed_surges": fn,
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "recall": tp / positives if positives else np.nan,
            "f1": 2 * tp / (2 * tp + fp + fn) if positives else np.nan}


weeks = pd.date_range(validation.week_start.min(), validation.week_start.max(), freq="W-MON")
alerts = validation.loc[validation.model_score.ge(cutoff)].sort_values(
    ["week_start", "model_score"] + cfg.GROUP_KEYS, ascending=[True, False] + [True] * len(cfg.GROUP_KEYS))
weekly_rank = alerts.groupby("week_start").cumcount()
summary_rows, weekly_rows, group_rows = [], [], []
for capacity in [None, 25, 50, 100]:
    name = "Frozen cutoff: all alerts" if capacity is None else f"At most {capacity} alerts/week"
    selected = alerts.index if capacity is None else alerts.index[weekly_rank.lt(capacity)]
    predicted = pd.Series(False, index=validation.index)
    predicted.loc[selected] = True
    counts = validation.loc[predicted].groupby("week_start").size().reindex(weeks, fill_value=0)
    summary_rows.append({"policy": name, "weekly_capacity": capacity, **policy_metrics(validation, predicted),
        "mean_cases_per_week": float(counts.mean()), "max_cases_per_week": int(counts.max())})
    for week, count in counts.items():
        weekly_rows.append({"policy": name, "week_start": week, "review_cases": int(count)})
    for group, part in validation.groupby("project_group"):
        group_rows.append({"policy": name, "project_group": group, **policy_metrics(part, predicted.loc[part.index])})
comparison = pd.DataFrame(summary_rows)
reference_path = checkpoint.parent / "validation_comparison.csv"
verify(reference_path, experiment["output_sha256"][reference_path.name])
reference = pd.read_csv(reference_path).set_index("version").loc["longer_2024_2025"]
for name in ["precision", "recall", "f1", "true_alerts", "false_alerts", "missed_surges"]:
    if not np.isclose(comparison.iloc[0][name], reference[name], rtol=1e-6, atol=1e-6, equal_nan=True):
        raise ValueError(f"Frozen reference {name} did not reproduce. Investigate before interpreting policies.")
ap = float(average_precision_score(validation.surge_label, validation.model_score))
if not np.isclose(ap, reference.average_precision, rtol=1e-6, atol=1e-6):
    raise ValueError("Frozen reference AP did not reproduce.")
display(comparison)
display(pd.DataFrame(group_rows))
print("Unchanged model AP:", round(ap, 6))
print("No policy was selected or applied. Lower workload may sacrifice recall; precision improvement is not guaranteed.")


,policy,weekly_capacity,rows,positives,review_cases,true_alerts,false_alerts,missed_surges,precision,recall,f1,mean_cases_per_week,max_cases_per_week
0,Frozen cutoff: all alerts,NaN,43594,3912,11361,2702,8659,1210,0.237831,0.690695,0.353827,946.75,1283
1,At most 25 alerts/week,25.0,43594,3912,300,168,132,3744,0.560000,0.042945,0.079772,25.00,25
2,At most 50 alerts/week,50.0,43594,3912,600,320,280,3592,0.533333,0.081800,0.141844,50.00,50
3,At most 100 alerts/week,100.0,43594,3912,1200,594,606,3318,0.495000,0.151840,0.232394,100.00,100


,policy,project_group,rows,positives,review_cases,true_alerts,false_alerts,missed_surges,precision,recall,f1
0,Frozen cutoff: all alerts,Chicken,4884,25,25,7,18,18,0.280000,0.280000,0.280000
1,Frozen cutoff: all alerts,Cooking Oil,2178,1,0,0,0,1,NaN,0.000000,0.000000
2,Frozen cutoff: all alerts,Eggs,8892,53,82,7,75,46,0.085366,0.132075,0.103704
3,Frozen cutoff: all alerts,Flour,1110,15,8,1,7,14,0.125000,0.066667,0.086957
4,Frozen cutoff: all alerts,Rice,384,2,2,1,1,1,0.500000,0.500000,0.500000
5,Frozen cutoff: all alerts,Sugar,1837,0,0,0,0,0,NaN,NaN,NaN
6,Frozen cutoff: all alerts,Vegetables,24309,3816,11244,2686,8558,1130,0.238883,0.703878,0.356707
7,At most 25 alerts/week,Chicken,4884,25,0,0,0,25,NaN,0.000000,0.000000
8,At most 25 alerts/week,Cooking Oil,2178,1,0,0,0,1,NaN,0.000000,0.000000
9,At most 25 alerts/week,Eggs,8892,53,0,0,0,53,NaN,0.000000,0.000000


Unchanged model AP: 0.329957
No policy was selected or applied. Lower workload may sacrifice recall; precision improvement is not guaranteed.


## 4. Save traceable, separate reports

Only derived files under `14_review_readiness/` are written. The original inputs, model, dashboard exports, source prices, labels and registration stay unchanged. A new fingerprint keeps changed inputs/code separate. The source log stays outside those fingerprint folders so handwritten evidence is not lost.


In [5]:
inputs = {str(path.resolve()): file_hash(path) for path in [entry_path, experiment_path, prediction_path,
    reference_path, folder / "experiment_settings.json", folder / "validation.parquet", review_path, log_path]}
notebook = json.loads(cfg.notebook_path("14_review_readiness.ipynb").read_text())
code_text = "\n".join("".join(cell["source"]) for cell in notebook["cells"] if cell["cell_type"] == "code")
protocol = {"status": "post-hoc exploratory validation-only review readiness",
    "forecast_model_sha256": model["sha256"], "cutoff": cutoff, "capacities": [25, 50, 100],
    "input_sha256": inputs, "code_sha256": hashlib.sha256(code_text.encode()).hexdigest(),
    "source_snapshot_sha256": snapshot, "review_decisions": "user-recorded; not independently verified by notebook",
    "selection_rule": "none; stakeholder capacity and prospective evaluation required before adoption"}
fingerprint = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()[:16]
output = cfg.PROJECT_ROOT / "14_review_readiness" / fingerprint
output.mkdir(parents=True, exist_ok=True)
comparison.to_csv(output / "policy_comparison.csv", index=False)
pd.DataFrame(weekly_rows).to_csv(output / "weekly_workload.csv", index=False)
pd.DataFrame(group_rows).to_csv(output / "policy_by_group.csv", index=False)
if any(file_hash(Path(name)) != expected for name, expected in inputs.items()):
    raise ValueError("An input changed during the audit; these reports are incomplete.")
record = {"protocol": protocol, "source_cases": len(review), "shortlisted_cases": len(shortlist),
    "log_decisions": log.decision.value_counts().to_dict(), "unchanged_average_precision": ap,
    "test_rows_loaded": False, "model_fitted": False, "dashboard_changed": False, "registration_changed": False,
    "output_sha256": {name: file_hash(output / name) for name in ["policy_comparison.csv", "weekly_workload.csv", "policy_by_group.csv"]}}
temporary = output / "review_manifest.tmp"
temporary.write_text(json.dumps(record, indent=2))
temporary.replace(output / "review_manifest.json")
print("Reports:", output)
print("Independent evidence log:", log_path)


Reports: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/14_review_readiness/e22ff4a3ed41e7ae
Independent evidence log: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/14_review_readiness/source_review/1866ba0fc10bebdc/source_review_log.csv


## What to do with these results

- **Source review:** unresolved is a valid finding. A suspicious value, neighbour agreement or a model-performance gain does not prove a source error. Independently supported corrections require a separately documented revised experiment; preserve the existing evidence and registration.
- **Workload:** ask intended reviewers how many cases per week they can investigate. Compare precision and recall at that workload. Do not pick the best-looking row from exposed outcomes and call it a confirmed improvement.
- **User evaluation:** ask users to find one item/location, explain a review alert versus an anomaly flag, check its price unit/dates/coverage, and identify missing support or explanations. Record task success, time, misunderstandings and whether the proposed review volume is manageable. This evaluates usability/usefulness, not actual affordability impact.
- Keep the current dashboard cutoff and the registered November comparison unchanged. New policy development remains exploratory until assessed on a genuinely untouched period under fixed rules.
